# 4.1 — Clasificacion multiclase con etiquetas de cluster no supervisadas

Replica simplificada del notebook 4.0, usando los datos enriquecidos con
etiquetas de cluster (`nm_cluster`) generados en el notebook 1.2.

**Cambio clave**: nonMalignant se subdivide en `nonMalignant_0` y `nonMalignant_1`
segun el cluster no supervisado, dando **20 clases** en lugar de 19.

Ranking:
1. Minimizar `test_cancer_fn`
2. Minimizar `test_cancer_fnr`
3. Maximizar `test_f1_macro`

In [2]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import MulticlassTrainConfig, run_training

warnings.filterwarnings("ignore", category=ConvergenceWarning)
logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*A worker stopped while some jobs were given to the executor.*",
)

### Rutas y carga de datos (con clusters)

In [3]:
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train_clustered.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test_clustered.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5453), (471, 5453))

### Separacion genes vs metadatos

In [4]:
metadata_cols = [
    "Sample ID", "Patient_group", "Stage", "Sex", "Age",
    "Sample-supplying institution", "Training series",
    "Evaluation series", "Validation series", "lib.size",
    "classificationScoreCancer", "Class_group", "nm_cluster",
]

gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "nm_cluster" in df_train.columns, "Ejecutar notebook 1.2 primero"
assert len(gene_cols) > 0

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check: distribucion de 20 clases

In [5]:
from genomics_dl.models.train_multiclass import build_y_multiclass_with_clusters

y_train_20 = build_y_multiclass_with_clusters(
    df_train,
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    cluster_col="nm_cluster",
)

print(f"Clases unicas: {len(np.unique(y_train_20))}")
print(pd.Series(y_train_20).value_counts())

Clases unicas: 20
Non-small-cell lung cancer    417
nonMalignant_0                385
nonMalignant_1                193
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64


## Sweep simplificado (20 configuraciones)

In [6]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [ ]:
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_clustered",
    model_version="v0.1.0",
    use_pca=False,
    var_quantile=0.2,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=5,
    min_cancer_recall_for_threshold=0.9,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_clustered",
    save_local_bundle=False,
    save_plots=False,
    cluster_col="nm_cluster",  # activa las 20 clases
)

# Grid simplificado basado en mejores configuraciones de 4.0
feat_grid = [
    dict(use_pca=False, selector_on_log=False, var_quantile=0.15, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=False, var_quantile=0.20, pca_var_threshold=0.9),
]

clf_grid = [
    ("rf", dict(n_estimators=500, max_depth=None)),
    ("rf", dict(n_estimators=1000, max_depth=None)),
    ("extratrees", dict(n_estimators=1000, max_depth=None, min_samples_leaf=2)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=1.0)),
]

malignant_weights = [3.0, 6.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(
                feat_cfg=feat_cfg, clf_name=clf_name,
                clf_params=clf_params, mw=mw,
            ))

print(f"Total combinaciones: {len(sweep)}")

Total combinaciones: 16


In [8]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep clustered", unit="run")

for combo in pbar:
    t0 = perf_counter()

    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="clust")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    try:
        out = run_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "var_quantile": feat_cfg["var_quantile"],
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_specificity": tm["cancer_specificity"],
            "test_f1_macro": tm["f1_macro"],
            "test_accuracy": tm["accuracy"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "test_cancer_roc_auc": tm["cancer_roc_auc"],
            "test_cancer_pr_auc": tm["cancer_pr_auc"],
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "var_quantile": feat_cfg["var_quantile"],
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt), "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed), "eta": fmt_secs(remaining),
        "ok": len(results), "err": len(errors),
    })

res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_f1_macro"],
                   ascending=[True, True, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

Sweep clustered: 100%|██████████| 16/16 [17:23<00:00, 65.23s/run, last=36s, avg=57s, elapsed=17m 23s, eta=0s, ok=16, err=0]          

OK: 16, Errores: 0


In [9]:
display(res_df)

,model_name,clf_name,mw,var_quantile,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_cancer_specificity,test_f1_macro,test_accuracy,test_balanced_accuracy,test_cancer_roc_auc,test_cancer_pr_auc,mlflow_run_id
0,rf_pca0_log0_vq15_mw6p0_clust,rf,6.0,0.15,38,0.116564,0.883436,0.558621,0.231069,0.439490,0.216686,0.833330,0.914224,5837e752dc934720b742460665e9b706
1,rf_pca0_log0_vq20_mw3p0_clust,rf,3.0,0.20,38,0.116564,0.883436,0.600000,0.230340,0.450106,0.219126,0.838492,0.917166,1276d5cd02a24fe89529fef24182db94
2,logreg_pca0_log0_vq20_mw6p0_clust,logreg,6.0,0.20,40,0.122699,0.877301,0.627586,0.383620,0.522293,0.391665,0.864713,0.940687,810513c9d2214af9a911c2f34674d8d0
3,logreg_pca0_log0_vq15_mw6p0_clust,logreg,6.0,0.15,40,0.122699,0.877301,0.634483,0.360126,0.518047,0.370783,0.859298,0.936454,178ba4ef4e8f4de89ffbf2bedc54a795
4,rf_pca0_log0_vq15_mw3p0_clust,rf,3.0,0.15,40,0.122699,0.877301,0.558621,0.223156,0.435244,0.214151,0.828580,0.913358,7263e5615b344a02b5a01b7fcece577f
5,rf_pca0_log0_vq20_mw3p0_clust,rf,3.0,0.20,41,0.125767,0.874233,0.586207,0.227596,0.441614,0.217696,0.836165,0.917028,65ff4c35601b49e89d7ad24c9a65df5a
6,rf_pca0_log0_vq15_mw6p0_clust,rf,6.0,0.15,41,0.125767,0.874233,0.551724,0.225502,0.435244,0.211494,0.830231,0.911530,a7e3759984194df7b8e228c971f27f23
7,rf_pca0_log0_vq20_mw6p0_clust,rf,6.0,0.20,41,0.125767,0.874233,0.558621,0.213925,0.424628,0.201563,0.837762,0.917227,541b537cd76744b7b2d8a910495e339c
8,extratrees_pca0_log0_vq15_mw3p0_clust,extratrees,3.0,0.15,41,0.125767,0.874233,0.572414,0.204504,0.430998,0.195244,0.824339,0.908138,d23f423917314a31a292a1899f0165ac
9,extratrees_pca0_log0_vq20_mw3p0_clust,extratrees,3.0,0.20,41,0.125767,0.874233,0.558621,0.182066,0.428875,0.184756,0.826560,0.909555,811af2b5365b40b7a403a7a19c13cbb7


In [10]:
if len(err_df) > 0:
    display(err_df)

## Entrenamiento final del mejor modelo

In [11]:
best = res_df.iloc[0].to_dict()
print("Mejor configuracion:")
for k, v in best.items():
    if k != "mlflow_run_id":
        print(f"  {k}: {v}")

Mejor configuracion:
  model_name: rf_pca0_log0_vq15_mw6p0_clust
  clf_name: rf
  mw: 6.0
  var_quantile: 0.15
  test_cancer_fn: 38
  test_cancer_fnr: 0.1165644171779141
  test_cancer_recall: 0.8834355828220859
  test_cancer_specificity: 0.5586206896551724
  test_f1_macro: 0.2310689162074278
  test_accuracy: 0.4394904458598726
  test_balanced_accuracy: 0.21668619873209088
  test_cancer_roc_auc: 0.8333298074888935
  test_cancer_pr_auc: 0.9142239790127418


In [12]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_clustered_final",
    model_version="v0.1.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    var_quantile=float(best["var_quantile"]),
    cv_splits=8,
    save_local_bundle=True,
    save_plots=True,
    output_figures_dir="reports/figures/multiclass_clustered",
)

final_out = run_training(best_cfg, feature_cols=gene_cols)
print(f"\nModelo guardado en: {final_out.get('bundle_dir', 'N/A')}")


Modelo guardado en: /workspaces/TFM/models/multiclass_clustered_final/v0.1.0


## Comparacion con notebook 4.0 (sin clusters)

In [13]:
# Resultados de referencia del notebook 4.0 (mejor modelo)
ref_40 = {
    "cancer_fn": 37,
    "cancer_fnr": 0.1135,
    "cancer_recall": 0.8865,
    "cancer_specificity": 0.5517,
    "f1_macro": 0.2037,
    "accuracy": 0.4565,
    "cancer_roc_auc": 0.8326,
    "cancer_pr_auc": 0.9141,
}

tm = final_out["test_metrics"]
res_41 = {
    "cancer_fn": tm["cancer_fn"],
    "cancer_fnr": round(tm["cancer_fnr"], 4),
    "cancer_recall": round(tm["cancer_recall_sensitivity"], 4),
    "cancer_specificity": round(tm["cancer_specificity"], 4),
    "f1_macro": round(tm["f1_macro"], 4),
    "accuracy": round(tm["accuracy"], 4),
    "cancer_roc_auc": round(tm["cancer_roc_auc"], 4),
    "cancer_pr_auc": round(tm["cancer_pr_auc"], 4),
}

comparison = pd.DataFrame({
    "4.0 (sin clusters)": ref_40,
    "4.1 (con clusters)": res_41,
})
comparison["delta"] = comparison["4.1 (con clusters)"] - comparison["4.0 (sin clusters)"]

display(comparison)

,4.0 (sin clusters),4.1 (con clusters),delta
cancer_fn,37.0000,41.0000,4.0000
cancer_fnr,0.1135,0.1258,0.0123
cancer_recall,0.8865,0.8742,-0.0123
cancer_specificity,0.5517,0.5655,0.0138
f1_macro,0.2037,0.2275,0.0238
accuracy,0.4565,0.4395,-0.0170
cancer_roc_auc,0.8326,0.8302,-0.0024
cancer_pr_auc,0.9141,0.9115,-0.0026
